# News Article Summarization - Reference Solution

Fine-tunes T5-small for abstractive news summarization. Evaluated with ROUGE-1, ROUGE-2, and ROUGE-L.

In [ ]:
import csv, torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.optim import AdamW

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL).to(DEVICE)
print(f'Model loaded on {DEVICE}')

In [ ]:
def load(path):
    with open(path,'r',encoding='utf-8') as f: return list(csv.DictReader(f))

train_rows = load('dataset/public/train.csv')
test_rows = load('dataset/public/test.csv')
print(f'Train: {len(train_rows)} | Test: {len(test_rows)}')

class SumDS(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = tokenizer('summarize: '+row['article_text'], truncation=True, max_length=512, padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labeled:
            with tokenizer.as_target_tokenizer():
                tgt = tokenizer(row['summary'], truncation=True, max_length=80, padding='max_length', return_tensors='pt')
            lbl = tgt['input_ids'].squeeze(0)
            lbl[lbl == tokenizer.pad_token_id] = -100
            item['labels'] = lbl
        return item

train_dl = DataLoader(SumDS(train_rows), batch_size=8, shuffle=True)
opt = AdamW(model.parameters(), lr=3e-4)

In [ ]:
model.train()
for epoch in range(3):
    loss_sum = 0
    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward(); opt.step(); opt.zero_grad()
        loss_sum += loss.item()
    print(f'Epoch {epoch+1} loss: {loss_sum/len(train_dl):.4f}')

In [ ]:
model.eval()
results = []
for row in test_rows:
    enc = tokenizer('summarize: '+row['article_text'], return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        ids = model.generate(**enc, max_new_tokens=80, num_beams=4, early_stopping=True)
    results.append({'article_id': row['article_id'], 'predicted_summary': tokenizer.decode(ids[0], skip_special_tokens=True)})

with open('predictions.csv','w',newline='') as f:
    w = csv.DictWriter(f,fieldnames=['article_id','predicted_summary'])
    w.writeheader(); w.writerows(results)
print(f'Saved {len(results)} summaries')